# B2S 04 - AndinaLog IoT Telemetry

Conversion auditada de lecturas IoT desde Bronze a Silver y cuarentena. Las entradas se leen sin modificarse. WMS Orders, Productos y Flota Silver se consultan exclusivamente para validar claves y reglas de negocio.

**Entidad:** lectura de telemetria de cabina. **Granularidad:** una fila por `viaje_id` y `timestamp` normalizados. Las desviaciones termicas operacionales no son errores de calidad por si mismas.

In [1]:
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 150)


def find_root():
    candidates = []
    if os.getenv("ANDINALOG_ROOT"):
        candidates.append(Path(os.environ["ANDINALOG_ROOT"]))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "datos" / "bronze").is_dir():
            return candidate
    raise FileNotFoundError("No se encontro el directorio datos/bronze")


ROOT = find_root()
EXECUTED_AT_UTC = datetime.now(timezone.utc).isoformat()
CONFIG = {
    "entidad": "lectura_iot_cabina",
    "granularidad": "una fila por viaje_id y timestamp normalizados",
    "clave": ["viaje_id", "timestamp"],
    "rutas": {
        "bronze": "datos/bronze/andinalog_iot_telemetry.csv",
        "wms_silver": "datos/silver/andinalog_wms_orders_silver.csv",
        "productos_silver": "datos/silver/andinalog_productos_silver.csv",
        "flota_silver": "datos/silver/andinalog_flota_silver.csv",
        "notebook": "notebooks/bronze_silver/04_iot_telemetry/B2S_04_AndinaLog_IoT_Telemetry.ipynb",
        "silver": "datos/silver/andinalog_iot_telemetry_silver.csv",
        "quarantine": "datos/quarantine/andinalog_iot_telemetry_quarantine.csv",
        "informe": "informes/bronze_silver/Informe_B2S_04_IoT_Telemetry.md",
    },
    "lectura": {"encoding": "utf-8", "dtype": "str", "keep_default_na": False},
    "columnas": {
        "obligatorias": [
            "timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
            "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct",
            "desviacion_termica_flag", "desviacion_proximos_60min_flag",
        ],
        "identificadores": ["viaje_id", "order_id", "camion_id", "producto_id"],
        "numericas": ["temperatura_cabina_c", "humedad_cabina_pct"],
        "binarias": ["desviacion_termica_flag", "desviacion_proximos_60min_flag"],
        "fecha": "timestamp",
        "unidad_temperatura": "temp_unit",
        "permitir_extra": False,
    },
    "formatos_identificador": {
        "viaje_id": r"^VIA-\d{5}$",
        "order_id": r"^ORD-2026-\d{5}$",
        "camion_id": r"^CAM-\d{2}$",
        "producto_id": r"^PROD-\d{3}$",
    },
    "formato_fecha": {"regex": r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$", "format": "%Y-%m-%d %H:%M:%S"},
    "zonas_horarias": {"sin_zona": "America/La_Paz", "silver": "UTC"},
    "unidades_temperatura": {"canonica": "C", "convertible": "F", "no_esperada": "K"},
    "conversion_fahrenheit_celsius": "(F - 32) * 5 / 9",
    "centinelas": {"temperatura_cabina_c": [-999], "humedad_cabina_pct": [-999]},
    "rangos_intrinsecos": {"humedad_cabina_pct": {"min": 0, "max": 100}},
    "rangos_termicos": {
        "aplicar_limite_tecnico": False,
        "motivo": "el plan propuso umbrales sin contrato; una desviacion operacional valida no es error de calidad",
    },
    "politica_duplicados": {"metodo": "cuarentena_de_todas_las_ocurrencias", "clave": ["viaje_id", "timestamp"]},
    "integridad_referencial": {
        "order_id": {"fuente": "wms_silver", "sin_correspondencia": "bandera_informativa"},
        "producto_id": {"fuente": "productos_silver", "sin_correspondencia": "bandera_informativa"},
        "camion_id": {"fuente": "flota_silver", "sin_correspondencia": "bandera_informativa"},
        "contradiccion_con_orden_wms": "error_bloqueante",
    },
    "imputaciones": {
        "habilitadas": False,
        "metodo": "",
        "motivo": "no existe una regla inequivoca; no se usan media, mediana, ffill, bfill ni lecturas futuras",
    },
    "semilla": None,
    "contradicciones_plan": [
        "hay 80 temperaturas vacias y 120 centinelas -999.0; el plan habia indicado ausencia de ambos",
        "hay 100 humedades vacias, no 951",
        "hay 15 fechas imposibles aunque todas cumplen el patron textual",
        "hay 230 filas duplicadas exactas y 240 filas en 120 claves de lectura duplicadas",
        "la frecuencia regular observada es 30 minutos, no 20-21",
        "no se aplican limites termicos -50/50 o -10/40 porque no existe contrato que los sustente",
        "las faltas referenciales se mantienen informativas, no bloqueantes, por no ser errores intrinsecos de sensor",
    ],
}
PATHS = {name: ROOT / relative for name, relative in CONFIG["rutas"].items()}
print("Raiz:", ROOT)
print("Fecha de ejecucion UTC:", EXECUTED_AT_UTC)
print("Contradicciones documentadas:", len(CONFIG["contradicciones_plan"]))

Raiz: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2
Fecha de ejecucion UTC: 2026-09-24T18:45:40.652908+00:00
Contradicciones documentadas: 7


In [2]:
bronze = pd.read_csv(PATHS["bronze"], **CONFIG["lectura"])
wms_silver = pd.read_csv(PATHS["wms_silver"], **CONFIG["lectura"])
productos_silver = pd.read_csv(PATHS["productos_silver"], **CONFIG["lectura"])
flota_silver = pd.read_csv(PATHS["flota_silver"], **CONFIG["lectura"])

parsed_profile_date = pd.to_datetime(bronze["timestamp"], format=CONFIG["formato_fecha"]["format"], errors="coerce")
numeric_profile = {
    column: pd.to_numeric(bronze[column], errors="coerce")
    for column in CONFIG["columnas"]["numericas"] + CONFIG["columnas"]["binarias"]
}
normalized_key = pd.DataFrame({
    "viaje_id": bronze["viaje_id"].str.strip().str.upper(),
    "timestamp": bronze["timestamp"].str.strip(),
})
profile_sorted = bronze.assign(
    _viaje=bronze["viaje_id"].str.strip().str.upper(),
    _timestamp=parsed_profile_date,
).sort_values(["_viaje", "_timestamp"])
profile_intervals = profile_sorted.groupby("_viaje")["_timestamp"].diff().dt.total_seconds().div(60)
perfil = {
    "filas": len(bronze),
    "columnas": len(bronze.columns),
    "nombres_columnas": bronze.columns.tolist(),
    "tipos_recibidos": bronze.dtypes.astype(str).to_dict(),
    "vacios": bronze.eq("").sum().to_dict(),
    "duplicados_exactos_filas": int(bronze.duplicated(keep=False).sum()),
    "duplicados_lectura_filas": int(normalized_key.duplicated(keep=False).sum()),
    "duplicados_lectura_claves": int(normalized_key.loc[normalized_key.duplicated(keep=False)].drop_duplicates().shape[0]),
    "espacios_identificadores": {
        column: int(bronze[column].ne(bronze[column].str.strip()).sum())
        for column in CONFIG["columnas"]["identificadores"]
    },
    "unidades_observadas": bronze["temp_unit"].value_counts(dropna=False).to_dict(),
    "centinelas": {
        column: int(numeric_profile[column].isin(values).sum())
        for column, values in CONFIG["centinelas"].items()
    },
    "fechas_formato_reconocido": int(bronze["timestamp"].str.match(CONFIG["formato_fecha"]["regex"]).sum()),
    "fechas_invalidas": int(parsed_profile_date.isna().sum()),
    "frecuencia_minutos": profile_intervals.value_counts().sort_index().to_dict(),
    "lecturas_por_viaje": bronze.groupby(bronze["viaje_id"].str.strip().str.upper()).size().value_counts().sort_index().to_dict(),
    "rangos_numericos_raw": {
        column: [float(series.min()), float(series.max())]
        for column, series in numeric_profile.items()
    },
}
print("Perfil Bronze")
for key, value in perfil.items():
    print(f"{key}: {value}")

Perfil Bronze
filas: 28920
columnas: 10
nombres_columnas: ['timestamp', 'viaje_id', 'order_id', 'camion_id', 'producto_id', 'temperatura_cabina_c', 'temp_unit', 'humedad_cabina_pct', 'desviacion_termica_flag', 'desviacion_proximos_60min_flag']
tipos_recibidos: {'timestamp': 'str', 'viaje_id': 'str', 'order_id': 'str', 'camion_id': 'str', 'producto_id': 'str', 'temperatura_cabina_c': 'str', 'temp_unit': 'str', 'humedad_cabina_pct': 'str', 'desviacion_termica_flag': 'str', 'desviacion_proximos_60min_flag': 'str'}
vacios: {'timestamp': 0, 'viaje_id': 0, 'order_id': 0, 'camion_id': 0, 'producto_id': 0, 'temperatura_cabina_c': 80, 'temp_unit': 0, 'humedad_cabina_pct': 100, 'desviacion_termica_flag': 0, 'desviacion_proximos_60min_flag': 0}
duplicados_exactos_filas: 230
duplicados_lectura_filas: 240
duplicados_lectura_claves: 120
espacios_identificadores: {'viaje_id': 0, 'order_id': 0, 'camion_id': 50, 'producto_id': 0}
unidades_observadas: {'C': 28865, 'F': 50, 'K': 5}
centinelas: {'temperat

In [3]:
def append_reason(df, mask, column, reason):
    df.loc[mask, column] = df.loc[mask, column].map(
        lambda current: reason if not current else f"{current} | {reason}"
    )
    return df


def validar_contrato_entrada(df):
    df = df.copy()
    expected = set(CONFIG["columnas"]["obligatorias"])
    received = set(df.columns)
    missing = sorted(expected - received)
    extra = sorted(received - expected)
    if missing or (extra and not CONFIG["columnas"]["permitir_extra"]):
        raise ValueError(f"Contrato invalido; faltantes={missing}, extra={extra}")
    return df


def estructurar(df):
    df = df.copy()
    df["_fila_bronze"] = range(2, len(df) + 2)
    for column in ["errores_bloqueantes", "banderas_informativas", "motivos_transformacion", "motivos_imputacion"]:
        df[column] = ""
    for column in CONFIG["columnas"]["obligatorias"]:
        df[f"{column}_original"] = df[column]
    return df


def normalizar_identificadores_y_unidad(df):
    df = df.copy()
    for column in CONFIG["columnas"]["identificadores"]:
        df[f"{column}_tratado"] = df[column].str.strip().str.upper()
        df[f"{column}_transformado"] = df[column].ne(df[f"{column}_tratado"])
        append_reason(df, df[f"{column}_transformado"], "motivos_transformacion", f"normalizacion_identificador:{column}")
    df["temp_unit_tratado"] = df["temp_unit"].str.strip().str.upper()
    df["temp_unit_transformado"] = df["temp_unit"].ne(df["temp_unit_tratado"])
    append_reason(df, df["temp_unit_transformado"], "motivos_transformacion", "normalizacion_unidad_temperatura")
    return df


def convertir_numericos_y_temperatura(df):
    df = df.copy()
    for column in CONFIG["columnas"]["numericas"] + CONFIG["columnas"]["binarias"]:
        raw_numeric = pd.to_numeric(df[column], errors="coerce")
        df[f"{column}_conversion_invalida"] = raw_numeric.isna() & df[column].ne("")
        append_reason(df, df[f"{column}_conversion_invalida"], "errores_bloqueantes", f"conversion_invalida:{column}")
        sentinels = CONFIG["centinelas"].get(column, [])
        df[f"{column}_centinela_detectado"] = raw_numeric.isin(sentinels)
        append_reason(df, df[f"{column}_centinela_detectado"], "errores_bloqueantes", f"centinela:{column}")
        df[f"{column}_tratado"] = raw_numeric.mask(df[f"{column}_centinela_detectado"])

    unit = df["temp_unit_tratado"]
    raw_temp = df["temperatura_cabina_c_tratado"].copy()
    is_c = unit.eq(CONFIG["unidades_temperatura"]["canonica"])
    is_f = unit.eq(CONFIG["unidades_temperatura"]["convertible"])
    is_k = unit.eq(CONFIG["unidades_temperatura"]["no_esperada"])
    known = is_c | is_f | is_k
    df["temperatura_convertida_fahrenheit"] = is_f & raw_temp.notna()
    df.loc[is_f, "temperatura_cabina_c_tratado"] = (raw_temp.loc[is_f] - 32) * 5 / 9
    append_reason(df, df["temperatura_convertida_fahrenheit"], "motivos_transformacion", "temperatura_fahrenheit_convertida_celsius")
    df["temperatura_unidad_no_esperada"] = is_k
    append_reason(df, is_k, "errores_bloqueantes", "unidad_temperatura_no_esperada:K")
    df.loc[is_k, "temperatura_cabina_c_tratado"] = pd.NA
    df["temperatura_unidad_desconocida"] = ~known
    append_reason(df, ~known, "errores_bloqueantes", "unidad_temperatura_desconocida")
    df.loc[~known, "temperatura_cabina_c_tratado"] = pd.NA
    df["temperatura_cabina_c_ausente"] = df["temperatura_cabina_c_original"].str.strip().eq("")
    append_reason(df, df["temperatura_cabina_c_ausente"], "errores_bloqueantes", "temperatura_ausente")

    humidity = df["humedad_cabina_pct_tratado"]
    df["humedad_cabina_pct_ausente"] = df["humedad_cabina_pct_original"].str.strip().eq("")
    append_reason(df, df["humedad_cabina_pct_ausente"], "banderas_informativas", "humedad_ausente_no_imputada")
    limits = CONFIG["rangos_intrinsecos"]["humedad_cabina_pct"]
    df["humedad_cabina_pct_fuera_rango"] = humidity.notna() & ~humidity.between(limits["min"], limits["max"])
    append_reason(df, df["humedad_cabina_pct_fuera_rango"], "errores_bloqueantes", "humedad_fuera_rango_0_100")
    return df


def convertir_timestamp(df):
    df = df.copy()
    rule = CONFIG["formato_fecha"]
    df["timestamp_formato_reconocido"] = df["timestamp"].str.match(rule["regex"])
    parsed = pd.to_datetime(df["timestamp"], format=rule["format"], errors="coerce")
    localized = parsed.dt.tz_localize(CONFIG["zonas_horarias"]["sin_zona"], ambiguous="NaT", nonexistent="NaT")
    df["timestamp_tratado"] = localized.dt.tz_convert(CONFIG["zonas_horarias"]["silver"])
    df["timestamp_conversion_invalida"] = df["timestamp_tratado"].isna()
    append_reason(df, df["timestamp_conversion_invalida"], "errores_bloqueantes", "timestamp_invalido")
    df["timestamp_transformado"] = ~df["timestamp_conversion_invalida"]
    append_reason(df, df["timestamp_transformado"], "motivos_transformacion", "fecha_local_bolivia_convertida_utc")
    return df


def validar_clave_secuencia_y_flags(df):
    df = df.copy()
    for column, pattern in CONFIG["formatos_identificador"].items():
        invalid = ~df[f"{column}_tratado"].fillna("").str.match(pattern)
        append_reason(df, invalid, "errores_bloqueantes", f"formato_invalido:{column}")
    duplicate = df.duplicated(["viaje_id_tratado", "timestamp_original"], keep=False)
    df["lectura_clave_duplicada"] = duplicate
    append_reason(df, duplicate, "errores_bloqueantes", "lectura_duplicada:viaje_id_timestamp")
    for column in CONFIG["columnas"]["binarias"]:
        invalid = df[f"{column}_tratado"].notna() & ~df[f"{column}_tratado"].isin([0, 1])
        append_reason(df, invalid, "errores_bloqueantes", f"indicador_no_binario:{column}")

    source_delta = df.groupby("viaje_id_tratado", sort=False)["timestamp_tratado"].diff().dt.total_seconds().div(60)
    df["intervalo_fuente_min"] = source_delta
    df["secuencia_fuente_fuera_orden"] = source_delta.lt(0)
    append_reason(df, df["secuencia_fuente_fuera_orden"], "banderas_informativas", "secuencia_fuente_fuera_orden")
    return df


def validar_integridad_referencial_y_operacional(df):
    df = df.copy()
    norm = lambda series: series.str.strip().str.upper()
    wms = wms_silver.assign(
        _order=norm(wms_silver["order_id"]),
        _producto=norm(wms_silver["producto_id"]),
        _camion=norm(wms_silver["camion_id"]),
    ).set_index("_order")
    product_keys = set(norm(productos_silver["producto_id"]))
    fleet_keys = set(norm(flota_silver["camion_id"]))
    df["order_id_corresponde_wms_silver"] = df["order_id_tratado"].isin(wms.index)
    df["producto_id_corresponde_silver"] = df["producto_id_tratado"].isin(product_keys)
    df["camion_id_corresponde_silver"] = df["camion_id_tratado"].isin(fleet_keys)
    append_reason(df, ~df["order_id_corresponde_wms_silver"], "banderas_informativas", "order_id_sin_correspondencia_wms_silver")
    append_reason(df, ~df["producto_id_corresponde_silver"], "banderas_informativas", "producto_id_sin_correspondencia_silver")
    append_reason(df, ~df["camion_id_corresponde_silver"], "banderas_informativas", "camion_id_sin_correspondencia_silver")

    expected_product = df["order_id_tratado"].map(wms["_producto"])
    expected_truck = df["order_id_tratado"].map(wms["_camion"])
    df["producto_id_coherente_con_wms"] = ~df["order_id_corresponde_wms_silver"] | df["producto_id_tratado"].eq(expected_product)
    df["camion_id_coherente_con_wms"] = ~df["order_id_corresponde_wms_silver"] | df["camion_id_tratado"].eq(expected_truck)
    append_reason(df, ~df["producto_id_coherente_con_wms"], "errores_bloqueantes", "producto_id_contradice_orden_wms")
    append_reason(df, ~df["camion_id_coherente_con_wms"], "errores_bloqueantes", "camion_id_contradice_orden_wms")

    products = productos_silver.assign(
        _producto=norm(productos_silver["producto_id"]),
        _temperatura=pd.to_numeric(productos_silver["temperatura_conservacion_requerida_c"], errors="coerce"),
        _tolerancia=pd.to_numeric(productos_silver["tolerancia_temperatura_c"], errors="coerce"),
    ).set_index("_producto")
    required = df["producto_id_tratado"].map(products["_temperatura"])
    tolerance = df["producto_id_tratado"].map(products["_tolerancia"])
    comparable = df["temperatura_cabina_c_tratado"].notna() & required.notna() & tolerance.notna()
    calculated = (
        df["temperatura_cabina_c_tratado"].lt(required - tolerance)
        | df["temperatura_cabina_c_tratado"].gt(required + tolerance)
    )
    df["desviacion_termica_calculable"] = comparable
    df["desviacion_termica_calculada"] = calculated.where(comparable).astype("boolean")
    observed = df["desviacion_termica_flag_tratado"].astype("Int64")
    df["desviacion_termica_flag_coherente"] = ~comparable | observed.eq(calculated.astype(int))
    append_reason(df, ~df["desviacion_termica_flag_coherente"], "errores_bloqueantes", "desviacion_termica_flag_incoherente")
    return df


def imputar(df):
    df = df.copy()
    df["fue_imputada"] = False
    df["imputacion_metodo"] = ""
    df["imputacion_motivo"] = ""
    return df


def asignar_calidad(df):
    df = df.copy()
    transform_flags = [f"{column}_transformado" for column in CONFIG["columnas"]["identificadores"]]
    transform_flags += ["temp_unit_transformado", "temperatura_convertida_fahrenheit", "timestamp_transformado"]
    df["fue_transformada"] = df[transform_flags].any(axis=1)
    df["conteo_transformaciones"] = df[transform_flags].sum(axis=1).astype(int)
    df["conteo_imputaciones"] = df["fue_imputada"].astype(int)
    df["conteo_errores_bloqueantes"] = df["errores_bloqueantes"].map(lambda value: len([x for x in value.split("|") if x.strip()]))
    df["conteo_banderas_informativas"] = df["banderas_informativas"].map(lambda value: len([x for x in value.split("|") if x.strip()]))
    df["calidad_motivo"] = df["errores_bloqueantes"].mask(df["errores_bloqueantes"].eq(""), "sin_errores_bloqueantes")
    df["calidad_estado"] = "valida"
    df.loc[df["fue_transformada"], "calidad_estado"] = "valida_con_transformacion"
    df.loc[df["errores_bloqueantes"].ne(""), "calidad_estado"] = "cuarentena"
    return df


work = (
    bronze.pipe(validar_contrato_entrada)
    .pipe(estructurar)
    .pipe(normalizar_identificadores_y_unidad)
    .pipe(convertir_numericos_y_temperatura)
    .pipe(convertir_timestamp)
    .pipe(validar_clave_secuencia_y_flags)
    .pipe(validar_integridad_referencial_y_operacional)
    .pipe(imputar)
    .pipe(asignar_calidad)
)
silver = work.loc[work["calidad_estado"].ne("cuarentena")].copy()
quarantine = work.loc[work["calidad_estado"].eq("cuarentena")].copy()
for frame in [silver, quarantine]:
    for column in CONFIG["columnas"]["identificadores"]:
        frame[column] = frame[f"{column}_tratado"]
    frame["timestamp"] = frame["timestamp_tratado"]
    frame["temperatura_cabina_c"] = frame["temperatura_cabina_c_tratado"]
    frame["temp_unit"] = CONFIG["unidades_temperatura"]["canonica"]
    nonconvertible_unit = frame["temperatura_unidad_no_esperada"] | frame["temperatura_unidad_desconocida"]
    frame.loc[nonconvertible_unit, "temp_unit"] = frame.loc[nonconvertible_unit, "temp_unit_tratado"]
    frame["humedad_cabina_pct"] = frame["humedad_cabina_pct_tratado"]
    for column in CONFIG["columnas"]["binarias"]:
        frame[column] = frame[f"{column}_tratado"]
silver.to_csv(PATHS["silver"], index=False, encoding="utf-8")
quarantine.to_csv(PATHS["quarantine"], index=False, encoding="utf-8")
print({"bronze": len(bronze), "silver": len(silver), "quarantine": len(quarantine)})

{'bronze': 28920, 'silver': 28448, 'quarantine': 472}


In [4]:
def reason_counts(series):
    counts = {}
    for value in series.fillna(""):
        for reason in [item.strip() for item in value.split("|") if item.strip()]:
            counts[reason] = counts.get(reason, 0) + 1
    return dict(sorted(counts.items(), key=lambda item: (-item[1], item[0])))


summary = {
    "unidades_observadas": bronze["temp_unit"].value_counts(dropna=False).to_dict(),
    "conversiones_fahrenheit_celsius": int(work["temperatura_convertida_fahrenheit"].sum()),
    "registros_kelvin": int(work["temperatura_unidad_no_esperada"].sum()),
    "centinelas_temperatura": int(work["temperatura_cabina_c_centinela_detectado"].sum()),
    "centinelas_humedad": int(work["humedad_cabina_pct_centinela_detectado"].sum()),
    "temperaturas_ausentes": int(work["temperatura_cabina_c_ausente"].sum()),
    "humedades_ausentes": int(work["humedad_cabina_pct_ausente"].sum()),
    "humedades_fuera_rango": int(work["humedad_cabina_pct_fuera_rango"].sum()),
    "desviaciones_termicas_operacionales": int(work["desviacion_termica_flag_tratado"].eq(1).sum()),
    "desviaciones_proximos_60min": int(work["desviacion_proximos_60min_flag_tratado"].eq(1).sum()),
    "sin_wms": int((~work["order_id_corresponde_wms_silver"]).sum()),
    "sin_producto_silver": int((~work["producto_id_corresponde_silver"]).sum()),
    "sin_flota_silver": int((~work["camion_id_corresponde_silver"]).sum()),
    "estados_calidad": work["calidad_estado"].value_counts().to_dict(),
    "errores_cuarentena": reason_counts(quarantine["errores_bloqueantes"]),
    "banderas_informativas": reason_counts(work["banderas_informativas"]),
    "silver": len(silver),
    "cuarentena": len(quarantine),
}
for key, value in summary.items():
    print(f"{key}: {value}")

unidades_observadas: {'C': 28865, 'F': 50, 'K': 5}
conversiones_fahrenheit_celsius: 50
registros_kelvin: 5
centinelas_temperatura: 120
centinelas_humedad: 0
temperaturas_ausentes: 80
humedades_ausentes: 100
humedades_fuera_rango: 15
desviaciones_termicas_operacionales: 951
desviaciones_proximos_60min: 1342
sin_wms: 4942
sin_producto_silver: 3159
sin_flota_silver: 1637
estados_calidad: {'valida_con_transformacion': 28448, 'cuarentena': 472}
errores_cuarentena: {'lectura_duplicada:viaje_id_timestamp': 240, 'centinela:temperatura_cabina_c': 120, 'temperatura_ausente': 80, 'humedad_fuera_rango_0_100': 15, 'timestamp_invalido': 15, 'unidad_temperatura_no_esperada:K': 5}
banderas_informativas: {'order_id_sin_correspondencia_wms_silver': 4942, 'producto_id_sin_correspondencia_silver': 3159, 'camion_id_sin_correspondencia_silver': 1637, 'secuencia_fuente_fuera_orden': 113, 'humedad_ausente_no_imputada': 100}
silver: 28448
cuarentena: 472


In [5]:
silver_file = pd.read_csv(PATHS["silver"], dtype=str, keep_default_na=False)
quarantine_file = pd.read_csv(PATHS["quarantine"], dtype=str, keep_default_na=False)
combined = pd.concat([silver_file, quarantine_file], ignore_index=True)
combined["_orden"] = pd.to_numeric(combined["_fila_bronze"])
combined = combined.sort_values("_orden").reset_index(drop=True)

assert len(bronze) == len(silver_file) + len(quarantine_file)
assert set(silver_file["_fila_bronze"]).isdisjoint(set(quarantine_file["_fila_bronze"]))
assert set(silver_file["_fila_bronze"]) | set(quarantine_file["_fila_bronze"]) == set(work["_fila_bronze"].astype(str))
assert not silver_file.duplicated(["viaje_id", "timestamp"], keep=False).any()
assert silver_file["errores_bloqueantes"].eq("").all()
assert silver_file["timestamp"].str.endswith("+00:00").all()
assert silver_file["temp_unit"].eq("C").all()
assert not silver_file["temperatura_unidad_no_esperada"].eq("True").any()
assert not silver_file["temperatura_cabina_c_centinela_detectado"].eq("True").any()
assert not silver_file["fue_imputada"].eq("True").any()
for column in CONFIG["columnas"]["obligatorias"]:
    assert combined[f"{column}_original"].equals(bronze[column].reset_index(drop=True))

report = f"""# Informe B2S 04 - AndinaLog IoT Telemetry

## Objetivo, entidad y granularidad
Conversion auditada de lecturas IoT desde Bronze a Silver y cuarentena.
- Entidad: {CONFIG["entidad"]}.
- Granularidad: {CONFIG["granularidad"]}.
- Clave de lectura: {CONFIG["clave"]}.

## Perfil Bronze
- Filas: {perfil["filas"]}; columnas: {perfil["columnas"]}.
- Columnas: {perfil["nombres_columnas"]}.
- Tipos recibidos: {perfil["tipos_recibidos"]}.
- Vacios: {perfil["vacios"]}.
- Duplicados exactos: {perfil["duplicados_exactos_filas"]} filas.
- Claves de lectura duplicadas: {perfil["duplicados_lectura_filas"]} filas en {perfil["duplicados_lectura_claves"]} claves.
- Espacios en identificadores: {perfil["espacios_identificadores"]}.
- Fechas invalidas: {perfil["fechas_invalidas"]}; frecuencia observada en minutos: {perfil["frecuencia_minutos"]}.

## Temperatura, humedad y unidades
- Unidades observadas: {summary["unidades_observadas"]}.
- Fahrenheit convertido exactamente a Celsius: {summary["conversiones_fahrenheit_celsius"]} filas.
- Kelvin no esperado enviado a cuarentena: {summary["registros_kelvin"]} filas.
- Centinelas de temperatura: {summary["centinelas_temperatura"]}; centinelas de humedad: {summary["centinelas_humedad"]}.
- Temperaturas ausentes: {summary["temperaturas_ausentes"]}; humedades ausentes conservadas sin imputar: {summary["humedades_ausentes"]}.
- Humedades fuera de 0-100: {summary["humedades_fuera_rango"]}.
- No se aplican limites termicos inventados. La desviacion se evalua contra temperatura requerida y tolerancia de Producto Silver cuando estan disponibles.

## Desviaciones operacionales e imputacion
- Desviaciones termicas observadas: {summary["desviaciones_termicas_operacionales"]}.
- Banderas de desviacion en proximos 60 minutos: {summary["desviaciones_proximos_60min"]}.
- Una desviacion operacional valida se conserva en Silver; no es un error de calidad.
- No se imputa ninguna variable. No se usan promedio, mediana, ffill, bfill ni lecturas futuras.

## Integridad referencial
- Filas sin WMS Silver: {summary["sin_wms"]}.
- Filas sin Producto Silver: {summary["sin_producto_silver"]}.
- Filas sin Flota Silver: {summary["sin_flota_silver"]}.
- Las faltas de correspondencia son banderas informativas, no errores intrinsecos de sensor. Una contradiccion con una orden WMS existente si es bloqueante.

## Contradicciones respecto del plan
""" + "\n".join(f"- {item}." for item in CONFIG["contradicciones_plan"]) + f"""

## Resultado y conciliacion
- Estados: {summary["estados_calidad"]}.
- Motivos de cuarentena: {summary["errores_cuarentena"]}.
- Banderas informativas: {summary["banderas_informativas"]}.
- Conciliacion: Bronze {len(bronze)} = Silver {len(silver_file)} + cuarentena {len(quarantine_file)}.
- Silver tiene clave de lectura unica, UTC, Celsius canonico y cero errores bloqueantes.
- Las columnas originales coinciden fila a fila con Bronze.

## Archivos generados
- `{CONFIG["rutas"]["notebook"]}`
- `{CONFIG["rutas"]["silver"]}`
- `{CONFIG["rutas"]["quarantine"]}`
- `{CONFIG["rutas"]["informe"]}`

## Reproducibilidad
- Fecha de ejecucion UTC: {EXECUTED_AT_UTC}.
- Python: {platform.python_version()}.
- pandas: {pd.__version__}.
- Rutas de entrada: `{CONFIG["rutas"]["bronze"]}`, `{CONFIG["rutas"]["wms_silver"]}`, `{CONFIG["rutas"]["productos_silver"]}`, `{CONFIG["rutas"]["flota_silver"]}`.
- Rutas de salida: `{CONFIG["rutas"]["silver"]}`, `{CONFIG["rutas"]["quarantine"]}`, `{CONFIG["rutas"]["informe"]}`.
- Zona inicial: {CONFIG["zonas_horarias"]["sin_zona"]}; zona Silver: {CONFIG["zonas_horarias"]["silver"]}.
- Conteos: Bronze {len(bronze)}, Silver {len(silver_file)}, cuarentena {len(quarantine_file)}.
- Semilla: {CONFIG["semilla"]} (no aplica; pipeline deterministico).
- Las entradas no se modifican; los controles finales se ejecutaron sobre CSV persistidos.
"""
PATHS["informe"].write_text(report, encoding="utf-8")
print("Controles persistidos: OK")
print(pd.DataFrame([{"bronze": len(bronze), "silver": len(silver_file), "quarantine": len(quarantine_file)}]))
print("Informe:", PATHS["informe"])

Controles persistidos: OK
   bronze  silver  quarantine
0   28920   28448         472
Informe: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2\informes\bronze_silver\Informe_B2S_04_IoT_Telemetry.md
